# DatetimeOutputParser → `date` / `datetime` 타입 필드

`DatetimeOutputParser`는 LangChain v1에서 **`langchain-classic`** 으로 이동한 레거시 API입니다. 또한 책의 `from langchain.prompts import PromptTemplate` 경로는 v1에서 제거되었으므로 `langchain_core.prompts`를 사용해야 합니다.

현재는 Pydantic 스키마의 필드 타입을 `datetime.date` / `datetime.datetime`으로 지정하면 됩니다.
- JSON Schema에 `"format": "date"` / `"date-time"`으로 전달되어 모델이 ISO 8601 형식으로 답하고,
- Pydantic이 이를 파이썬 `date` / `datetime` 객체로 자동 변환·검증합니다.

`output_parser.format = "%Y-%m-%d"`처럼 형식 문자열을 맞출 필요가 없습니다. 출력 형식은 결과 객체에 `strftime()`을 적용해 원하는 대로 바꾸면 됩니다.

In [ ]:
# 최초 1회 설치 (LangChain v1 기준)
# %pip install -qU langchain langchain-openai langchain-classic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 의 OPENAI_API_KEY, LANGSMITH_API_KEY 를 불러옵니다.

# LangSmith 추적: 별도 헬퍼 없이 환경변수만 설정하면 자동으로 활성화됩니다.
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "CH03-OutputParser")

In [ ]:
from langchain.chat_models import init_chat_model

# 공급자 중립적인 모델 초기화 ("공급자:모델명")
# 다른 모델로 바꾸려면 문자열만 교체하면 됩니다. 예) "anthropic:claude-sonnet-4-5", "ollama:llama3.1"
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

**참고: `strftime` 형식 코드**

| 형식 코드 | 설명                | 예시          |
|------------|---------------------|---------------|
| %Y         | 4자리 연도          | 2024          |
| %y         | 2자리 연도          | 24            |
| %m         | 2자리 월            | 07            |
| %d         | 2자리 일            | 04            |
| %H         | 24시간제 시간       | 14            |
| %I         | 12시간제 시간       | 02            |
| %p         | AM 또는 PM          | PM            |
| %M         | 2자리 분            | 45            |
| %S         | 2자리 초            | 08            |
| %f         | 마이크로초 (6자리)  | 000123        |
| %z         | UTC 오프셋          | +0900         |
| %Z         | 시간대 이름         | KST           |
| %a         | 요일 약어           | Thu           |
| %A         | 요일 전체           | Thursday      |
| %b         | 월 약어             | Jul           |
| %B         | 월 전체             | July          |
| %c         | 전체 날짜와 시간     | Thu Jul  4 14:45:08 2024 |
| %x         | 전체 날짜           | 07/04/24      |
| %X         | 전체 시간           | 14:45:08      |

## 1. 날짜(`date`) 필드

In [ ]:
from datetime import date, datetime

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class DateAnswer(BaseModel):
    """질문에 대한 날짜 답변"""

    value: date = Field(description="질문에 해당하는 날짜")


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the users question."),
        ("human", "{question}"),
    ]
)

chain = prompt | llm.with_structured_output(DateAnswer)

output = chain.invoke({"question": "Google 이 창업한 날짜"})
print(type(output.value), output.value)

결과는 이미 `date` 객체이므로 원하는 형식의 문자열로 바로 변환할 수 있습니다.

In [ ]:
print(output.value.strftime("%Y-%m-%d"))
print(output.value.strftime("%Y년 %m월 %d일 (%A)"))

## 2. 날짜 + 시각(`datetime`) 필드

필드를 여러 개 두면 날짜와 함께 부가 정보도 한 번에 받을 수 있습니다. 이것이 단일 값만 반환하던 `DatetimeOutputParser` 대비 장점입니다.

In [ ]:
class EventTime(BaseModel):
    """역사적 사건의 발생 시각"""

    event: str = Field(description="사건 이름")
    occurred_at: datetime = Field(description="사건 발생 시각 (UTC, ISO 8601)")


event_chain = prompt | llm.with_structured_output(EventTime)

result = event_chain.invoke({"question": "아폴로 11호가 달에 착륙한 시각"})
print(result.event)
print(result.occurred_at, "| tzinfo:", result.occurred_at.tzinfo)
print(result.occurred_at.strftime("%Y-%m-%d %H:%M %Z"))

## (참고) 레거시 API

`from langchain_classic.output_parsers import DatetimeOutputParser` 로 여전히 사용할 수 있지만, 신규 코드에는 권장하지 않습니다.